In [1]:
#Pandas import and folder path
import pandas as pd
deirectoryPath ="../data/"

In [2]:
#Reading dataset
dataDF = pd.read_csv(f"{deirectoryPath}NF-UNSW-NB15-v2.csv")

In [3]:
dataDF.head()

,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,...,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
0,59.166.0.5,1305,149.171.126.8,21,6,1.0,9,1,193,3,...,0,7240,0,0,0,0,0,331.0,0,Benign
1,59.166.0.5,1305,149.171.126.8,21,6,1.0,261,5,469,7,...,8688,8688,18944,74,0,0,0,230.0,0,Benign
2,59.166.0.5,1305,149.171.126.8,21,6,1.0,481,9,750,11,...,10136,10136,33792,132,0,0,0,229.0,0,Benign
3,59.166.0.5,1305,149.171.126.8,21,6,1.0,701,13,1054,15,...,11584,11584,48640,190,0,0,0,125.0,0,Benign
4,59.166.0.5,1305,149.171.126.8,21,6,1.0,1031,19,1474,21,...,14480,13032,64256,251,0,0,0,230.0,0,Benign


In [4]:
len(dataDF.columns)

45

In [5]:
len(dataDF['IPV4_SRC_ADDR'].unique())

40

In [6]:
len(dataDF['IPV4_DST_ADDR'].unique())

40

In [7]:
#Droping unwanted features
dataDF.drop(columns=["Attack" , "L4_SRC_PORT"] , inplace = True , axis = 1)
dataDF.head()

,IPV4_SRC_ADDR,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,...,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label
0,59.166.0.5,149.171.126.8,21,6,1.0,9,1,193,3,24,...,0,0,7240,0,0,0,0,0,331.0,0
1,59.166.0.5,149.171.126.8,21,6,1.0,261,5,469,7,24,...,0,8688,8688,18944,74,0,0,0,230.0,0
2,59.166.0.5,149.171.126.8,21,6,1.0,481,9,750,11,24,...,0,10136,10136,33792,132,0,0,0,229.0,0
3,59.166.0.5,149.171.126.8,21,6,1.0,701,13,1054,15,24,...,0,11584,11584,48640,190,0,0,0,125.0,0
4,59.166.0.5,149.171.126.8,21,6,1.0,1031,19,1474,21,24,...,0,14480,13032,64256,251,0,0,0,230.0,0


In [8]:
#Train test split(Based on edges)
from sklearn.model_selection import train_test_split

trainDF , testDF = train_test_split(dataDF , test_size=0.25 , random_state=20 , stratify=dataDF['Label'])

In [9]:
#Initializing train and test graph
import networkx as netx
trainingGraph = netx.MultiDiGraph()
testingGraph = netx.MultiDiGraph()

In [10]:
dataDF.columns

Index(['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO',
       'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS',
       'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS',
       'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT',
       'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
       'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES',
       'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS',
       'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS',
       'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
       'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
       'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
       'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
       'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE',
       'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Label'],
      dtype='object')

In [11]:
#Defining node based on IP
srcTrain = trainDF['IPV4_SRC_ADDR'].to_list()
destTrain = trainDF['IPV4_DST_ADDR'].to_list() 
srcTest = testDF['IPV4_SRC_ADDR'].to_list()
destTest = testDF['IPV4_DST_ADDR'].to_list() 

In [12]:
#Adding node to graphs
trainingGraph.add_nodes_from(srcTrain)
trainingGraph.add_nodes_from(destTrain)
testingGraph.add_nodes_from(srcTest)
testingGraph.add_nodes_from(destTest)

In [13]:
#Preparing col for features
feature_cols = [col for col in trainDF.columns if col not in ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label']]

In [14]:
#Splitting continuous values and categorical values and standarizing continuous values
from sklearn.preprocessing import StandardScaler
scalerEdge = StandardScaler()
train_feats = scalerEdge.fit_transform(trainDF[feature_cols].values.astype(float))
scalerEdge = StandardScaler()

cat_cols = [
    'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO',
    'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS',
    'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_TYPE',
    'FTP_COMMAND_RET_CODE'
]
cont_cols = [col for col in feature_cols if col not in cat_cols]

train_cont = trainDF[cont_cols].astype(float)
test_cont = testDF[cont_cols].astype(float)

train_feats_cont = pd.DataFrame(
    scalerEdge.fit_transform(train_cont),
    columns=cont_cols,
    index=trainDF.index
)
test_feats_cont = pd.DataFrame(
    scalerEdge.transform(test_cont),
    columns=cont_cols,
    index=testDF.index
)

train_feats = pd.concat(
    [train_feats_cont, trainDF[cat_cols].astype(float)],
    axis=1
)[feature_cols].to_numpy()

test_feats = pd.concat(
    [test_feats_cont, testDF[cat_cols].astype(float)],
    axis=1
)[feature_cols].to_numpy()

In [15]:
for i, (idx, row) in enumerate(trainDF.iterrows()):
    trainingGraph.add_edge(row['IPV4_SRC_ADDR'], row['IPV4_DST_ADDR'],
                            features=train_feats[i], label=row['Label'])

In [16]:
for i, (idx, row) in enumerate(testDF.iterrows()):
    testingGraph.add_edge(row['IPV4_SRC_ADDR'], row['IPV4_DST_ADDR'],
                            features=test_feats[i], label=row['Label'])

In [17]:
print(f"Training Graph - Nodes: {trainingGraph.number_of_nodes()}, Edges: {trainingGraph.number_of_edges()}")
print(f"Test Graph - Nodes: {testingGraph.number_of_nodes()}, Edges: {testingGraph.number_of_edges()}")
print(f"Edge features shape: {trainingGraph[list(trainingGraph.edges())[0][0]][list(trainingGraph.edges())[0][1]][0]['features'].shape}")

Training Graph - Nodes: 43, Edges: 1792706
Test Graph - Nodes: 42, Edges: 597569
Edge features shape: (40,)


In [18]:
src_stats_train = trainDF.groupby("IPV4_SRC_ADDR").agg(
    out_degree=("IPV4_SRC_ADDR", "count"),
    avg_sbytes=("IN_BYTES", "mean"),
    avg_spkts=("IN_PKTS", "mean")
)

dst_stats_train = trainDF.groupby("IPV4_DST_ADDR").agg(
    in_degree=("IPV4_DST_ADDR", "count"),
    avg_dbytes=("OUT_BYTES", "mean"),
    avg_dpkts=("OUT_PKTS", "mean")
)

In [19]:
src_stats_test = testDF.groupby("IPV4_SRC_ADDR").agg(
    out_degree=("IPV4_SRC_ADDR", "count"),
    avg_sbytes=("IN_BYTES", "mean"),
    avg_spkts=("IN_PKTS", "mean")
)

dst_stats_test = testDF.groupby("IPV4_DST_ADDR").agg(
    in_degree=("IPV4_DST_ADDR", "count"),
    avg_dbytes=("OUT_BYTES", "mean"),
    avg_dpkts=("OUT_PKTS", "mean")
)

In [20]:
nodeOrder = list(trainingGraph.nodes())
trainNodeFeature = pd.concat(
    [src_stats_train, dst_stats_train],
    axis=1
)

trainNodeFeature = trainNodeFeature.reindex(nodeOrder).fillna(0)
assert list(trainNodeFeature.index) == nodeOrder

In [21]:
nodeOrder = list(testingGraph.nodes())
testNodeFeature = pd.concat(
    [src_stats_test, dst_stats_test],
    axis=1
)

testNodeFeature = testNodeFeature.reindex(nodeOrder).fillna(0)
assert list(testNodeFeature.index) == nodeOrder

In [22]:
nodeScaler = StandardScaler()
trainNodeFeature[:] = nodeScaler.fit_transform(trainNodeFeature)

In [23]:
testNodeFeature[:] = nodeScaler.transform(testNodeFeature)

In [24]:
import torch
from torch_geometric.utils import from_networkx

trainGraphData = from_networkx(trainingGraph)
testGraphData = from_networkx(testingGraph)

/mnt/d/Personal_project/DeepLearning/nnForGraphs/attackClassifier/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/d/Personal_project/DeepLearning/nnForGraphs/attackClassifier/venv/lib/python3.10/site-packages/torch_geometric/utils/convert.py:278: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  data_dict[key] = torch.as_tensor(value)


In [25]:
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [26]:
trainGraphData.x = torch.tensor(trainNodeFeature.values, dtype=torch.float).to(device)
testGraphData.x = torch.tensor(testNodeFeature.values, dtype=torch.float).to(device)

In [27]:
edge_features_train = []
edge_labels_train = []

for _ , _ , attr in trainingGraph.edges(data=True):
    edge_features_train.append(attr['features'])
    edge_labels_train.append(attr['label'])

In [28]:
edge_features_test = []
edge_labels_test = []

for _ , _ , attr in testingGraph.edges(data=True):
    edge_features_test.append(attr['features'])
    edge_labels_test.append(attr['label'])

In [29]:
len(edge_features_train[0])

40

In [30]:
trainGraphData.edge_attr = torch.tensor(
    edge_features_train,
    dtype=torch.float
).to(device)

trainGraphData.edge_label = torch.tensor(
    edge_labels_train,
    dtype=torch.float
).to(device)

In [31]:
testGraphData.edge_attr = torch.tensor(
    edge_features_test,
    dtype=torch.float
).to(device)

testGraphData.edge_label = torch.tensor(
    edge_labels_test,
    dtype=torch.float
).to(device)

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as funct
from torch_geometric.nn import GINEConv

In [33]:
print(trainGraphData)

Data(edge_index=[2, 1792706], features=[1792706, 40], label=[1792706], num_nodes=43, x=[43, 6], edge_attr=[1792706, 40], edge_label=[1792706])


In [34]:
# shapes must line up: edge_attr rows == edge_index columns
assert trainGraphData.edge_attr.shape[0] == trainGraphData.edge_index.shape[1]
assert testGraphData.edge_attr.shape[0] == testGraphData.edge_index.shape[1]

# node feature rows must match node count
assert trainGraphData.x.shape[0] == trainGraphData.num_nodes
assert testGraphData.x.shape[0] == testGraphData.num_nodes

In [43]:
class EdgeGCN(nn.Module):
    def __init__(self, nodeFeatureDim , edgeFeatureDim , embeddingDim , labelDim):
        super().__init__()
        
        conv1nn = nn.Sequential(
            nn.Linear(nodeFeatureDim , 128),
            nn.ReLU(),
            nn.Linear(128 , 64)
        )
        conv2nn = nn.Sequential(
                nn.Linear(64 , 32),
                nn.ReLU(),
                nn.Linear(32 , 16)
        )
        conv3nn = nn.Sequential(
                nn.Linear(16 , 8),
                nn.ReLU(),
                nn.Linear(8 , embeddingDim)
            )
        self.gcnConvoLayer1 = GINEConv(nn=conv1nn , edge_dim=edgeFeatureDim)
        self.gcnConvoLayer2 = GINEConv(nn=conv2nn , edge_dim=edgeFeatureDim)
        self.gcnConvoLayer3 = GINEConv(nn=conv3nn , edge_dim=edgeFeatureDim)

        #Classifier
        self.classifierLinearModel = nn.Sequential(
            nn.Linear(2*embeddingDim + edgeFeatureDim , 128),
            nn.ReLU(),
            nn.Linear(128 , 64),
            nn.ReLU(),
            nn.Linear(64 , 32),
            nn.ReLU(),
            nn.Linear(32,labelDim)
        )

    def forward(self, x, edge_index, edge_label_index, edge_attr , edge_label_attr):
        x = self.gcnConvoLayer1(x, edge_index , edge_attr)
        x = funct.relu(x)
        x = self.gcnConvoLayer2(x, edge_index , edge_attr)
        x = funct.relu(x)
        x = self.gcnConvoLayer3(x, edge_index , edge_attr)

        src = x[edge_label_index[0]]
        dst = x[edge_label_index[1]]

        edgeRepresentation = torch.cat([src, dst, edge_label_attr], dim=1)
        out = self.classifierLinearModel(edgeRepresentation)
        return out


In [44]:
classifier = EdgeGCN(
    nodeFeatureDim=trainGraphData.x.shape[1],
    edgeFeatureDim=trainGraphData.edge_attr.shape[1],
    embeddingDim=16,
    labelDim=2
).to(device)

In [45]:
label_counts = trainDF['Label'].value_counts().sort_index()
print(label_counts)

total = label_counts.sum()
num_classes = len(label_counts)

weight = total / (num_classes * label_counts)
weight = torch.tensor(weight.values, dtype=torch.float).to(device)

Label
0    1721416
1      71290
Name: count, dtype: int64


In [46]:
optimizer = torch.optim.Adam(
    classifier.parameters(),
    lr = 0.001
)
criterion = nn.CrossEntropyLoss(weight=weight)

In [47]:
trainGraphData = trainGraphData.to(device)
testGraphData = testGraphData.to(device)

In [48]:
from torch_geometric.loader import LinkNeighborLoader

train_loader = LinkNeighborLoader(
    data=trainGraphData,
    num_neighbors=[10, 10, 10],          # one entry per conv layer
    edge_label_index=trainGraphData.edge_index,
    edge_label=trainGraphData.edge_label,
    batch_size=2048,
    shuffle=True,
)

test_loader = LinkNeighborLoader(
    data=testGraphData,
    num_neighbors=[10, 10, 10],
    edge_label_index=testGraphData.edge_index,
    edge_label=testGraphData.edge_label,
    batch_size=2048,
    shuffle=False,                        # no need to shuffle eval
)

In [49]:
print(batch.x.shape)
print(batch.edge_index.shape)
print(batch.edge_attr.shape)
print(batch.edge_label_index.shape)
print(batch.edge_label.shape)
print(batch.input_id.shape)

torch.Size([37, 6])
torch.Size([2, 350])
torch.Size([350, 40])
torch.Size([2, 2048])
torch.Size([2048])
torch.Size([2048])


In [52]:
epochs = 80
for epoch in range(epochs):
    classifier.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        edge_attr_batch = trainGraphData.edge_attr[batch.input_id].to(device)

        optimizer.zero_grad(set_to_none=True)
        out = classifier(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_label_index=batch.edge_label_index,
            edge_attr=batch.edge_attr,
            edge_label_attr=edge_attr_batch
        )
        loss = criterion(out, batch.edge_label.long())
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.edge_label.size(0)

    avg_loss = total_loss / trainGraphData.edge_index.size(1)
    print(f"Epoch {epoch+1:03d} Loss: {avg_loss:.4f}")

Epoch 001 Loss: 0.0298
Epoch 002 Loss: 0.0288
Epoch 003 Loss: 0.0292
Epoch 004 Loss: 0.0292
Epoch 005 Loss: 0.0292
Epoch 006 Loss: 0.0292
Epoch 007 Loss: 0.0703
Epoch 008 Loss: 0.0244
Epoch 009 Loss: 0.0814
Epoch 010 Loss: 0.0685
Epoch 011 Loss: 0.0264
Epoch 012 Loss: 0.0463
Epoch 013 Loss: 0.0269
Epoch 014 Loss: 0.0270
Epoch 015 Loss: 0.0609
Epoch 016 Loss: 0.0291
Epoch 017 Loss: 0.0744
Epoch 018 Loss: 0.0570
Epoch 019 Loss: 0.0325
Epoch 020 Loss: 0.0385
Epoch 021 Loss: 0.0235
Epoch 022 Loss: 0.0352
Epoch 023 Loss: 0.8489
Epoch 024 Loss: 0.0272
Epoch 025 Loss: 0.0244
Epoch 026 Loss: 0.0243
Epoch 027 Loss: 0.0234
Epoch 028 Loss: 0.0270
Epoch 029 Loss: 0.0234
Epoch 030 Loss: 0.0303
Epoch 031 Loss: 0.0235
Epoch 032 Loss: 0.0235
Epoch 033 Loss: 0.0235
Epoch 034 Loss: 0.0234
Epoch 035 Loss: 0.0235
Epoch 036 Loss: 0.0235
Epoch 037 Loss: 0.0235
Epoch 038 Loss: 0.0235
Epoch 039 Loss: 0.0235
Epoch 040 Loss: 4.5228
Epoch 041 Loss: 0.0408
Epoch 042 Loss: 0.0297
Epoch 043 Loss: 0.0265
Epoch 044 L

In [54]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np
import torch

classifier.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        edge_attr_batch = testGraphData.edge_attr[batch.input_id].to(device)

        out = classifier(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_label_index=batch.edge_label_index,
            edge_attr=batch.edge_attr,
            edge_label_attr=edge_attr_batch
        )
        preds = out.argmax(dim=1).cpu().numpy()
        trues = batch.edge_label.long().cpu().numpy()

        y_pred.append(preds)
        y_true.append(trues)

y_pred = np.concatenate(y_pred)
y_true = np.concatenate(y_true)

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, digits=4)

print(f"Accuracy: {acc:.4f}")
print("Confusion Matrix:")
print(cm)
print("Classification Report:")
print(report)

Accuracy: 0.9922
Confusion Matrix:
[[569147   4659]
 [     0  23763]]
Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9919    0.9959    573806
           1     0.8361    1.0000    0.9107     23763

    accuracy                         0.9922    597569
   macro avg     0.9180    0.9959    0.9533    597569
weighted avg     0.9935    0.9922    0.9925    597569

